In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(r'C:\Users\konta\Documents\DIV_Academy\Module2(From_29_nov)\data\housing.csv')
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [13]:
outliers1 = df['median_house_value'] == df['median_house_value'].max()
outliers2 = df['housing_median_age'] == df['housing_median_age'].max()
outliers = outliers1 | outliers2

In [14]:
outliers

0        False
1        False
2         True
3         True
4         True
         ...  
20635    False
20636    False
20637    False
20638    False
20639    False
Length: 20640, dtype: bool

##### Q1 = df[col].quantile(0.25)
##### Q3 = df[col].quantile(0.75)
##### IQR = Q3 - Q1

##### outliers = (df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)

In [15]:
df = df.loc[~outliers, :].copy()
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
8,-122.26,37.84,42.0,2555.0,665.0,1206.0,595.0,2.0804,226700.0,NEAR BAY
15,-122.26,37.85,50.0,1120.0,283.0,697.0,264.0,2.1250,140000.0,NEAR BAY
18,-122.26,37.84,50.0,2239.0,455.0,990.0,419.0,1.9911,158700.0,NEAR BAY
...,...,...,...,...,...,...,...,...,...,...
20635,-121.09,39.48,25.0,1665.0,374.0,845.0,330.0,1.5603,78100.0,INLAND
20636,-121.21,39.49,18.0,697.0,150.0,356.0,114.0,2.5568,77100.0,INLAND
20637,-121.22,39.43,17.0,2254.0,485.0,1007.0,433.0,1.7000,92300.0,INLAND
20638,-121.32,39.43,18.0,1860.0,409.0,741.0,349.0,1.8672,84700.0,INLAND


In [16]:
X = df.drop('median_house_value', axis='columns')
y = df['median_house_value']

In [17]:
X['income_cat'] = pd.cut(
    X['median_income'],
    bins=[0, 1.5, 3.0, 4.5, 6, np.inf],
    labels=[1, 2, 3, 4, 5]
)

In [18]:
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=X['income_cat'],
    test_size=0.2
)

In [19]:
X_train.drop('income_cat', axis='columns', inplace=True)
X_test.drop('income_cat', axis='columns', inplace=True)

In [21]:
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.cluster import KMeans

n_clusters = 10
kmeans_ = KMeans(n_clusters=n_clusters)
kmeans_.fit(X_train[['latitude', 'longitude']])
print(kmeans_.cluster_centers_)

[[  37.73519205 -120.95007947]
 [  33.92326273 -117.71629134]
 [  37.49793437 -122.05615902]
 [  38.32761158 -122.49121834]
 [  34.0833175  -118.3721825 ]
 [  36.08917936 -119.66179357]
 [  38.76997015 -121.22152239]
 [  32.98488235 -116.90904575]
 [  40.20052632 -121.85133127]
 [  40.68563636 -123.83751515]]


In [22]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10):
        self.n_clusters = n_clusters

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(self.n_clusters)
        self.kmeans_.fit(X)
        return self  

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=1).round(3)

    def get_feature_names_out(self, names=None):
        return [f"Cluster {i} similarity" for i in range(self.n_clusters)]

In [23]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

cat_cols = ['ocean_proximity']
log_cols = [
    'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income'
    ]

cat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])
log_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out='one-to-one')),
    ('scl', StandardScaler())
])

def ratio_func(arr_2d):
    return arr_2d[:, [0]] / arr_2d[:, [1]]
def ratio_feature_names(*args, **kwargs):
    return ['ratio']

rat_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('newcol', FunctionTransformer(ratio_func, feature_names_out=ratio_feature_names)),
    ('scl', StandardScaler())
])
cluster_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('cluster', ClusterSimilarity(n_clusters=10))
])
num_pipeline = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('scl', StandardScaler())
])

preprocessing = ColumnTransformer(
    transformers=[
        ('CAT', cat_pipeline, cat_cols),
        ('LOG', log_pipeline, log_cols),
        ('RAT_b/r', rat_pipeline, ['total_bedrooms', 'total_rooms']),
        ('RAT_r/h', rat_pipeline, ['total_rooms', 'households']),
        ('RAT_p/h', rat_pipeline, ['population', 'households']),
        ("GEO", cluster_pipeline, ["latitude", "longitude"]),
    ], remainder=num_pipeline
)

X_train_arr = preprocessing.fit_transform(X_train)
X_train_prepared = pd.DataFrame(X_train_arr, columns=preprocessing.get_feature_names_out())

X_test_arr = preprocessing.fit_transform(X_test)
X_test_prepared = pd.DataFrame(X_test_arr, columns=preprocessing.get_feature_names_out())

X_train_prepared

,CAT__ocean_proximity_<1H OCEAN,CAT__ocean_proximity_INLAND,CAT__ocean_proximity_ISLAND,CAT__ocean_proximity_NEAR BAY,CAT__ocean_proximity_NEAR OCEAN,LOG__total_rooms,LOG__total_bedrooms,LOG__population,LOG__households,LOG__median_income,...,GEO__Cluster 1 similarity,GEO__Cluster 2 similarity,GEO__Cluster 3 similarity,GEO__Cluster 4 similarity,GEO__Cluster 5 similarity,GEO__Cluster 6 similarity,GEO__Cluster 7 similarity,GEO__Cluster 8 similarity,GEO__Cluster 9 similarity,remainder__housing_median_age
0,0.0,1.0,0.0,0.0,0.0,1.621169,1.719222,1.325294,1.773328,-0.869780,...,0.001,0.000,0.000,0.926,0.000,0.000,0.007,0.088,0.000,-0.875960
1,0.0,0.0,0.0,0.0,1.0,-1.075289,-0.785448,-0.201637,-0.722394,-1.456528,...,0.000,0.166,0.000,0.000,0.000,0.061,0.000,0.000,0.956,-0.176914
2,0.0,1.0,0.0,0.0,0.0,0.335006,0.287085,-0.013065,0.379782,-0.668251,...,0.431,0.000,0.027,0.000,0.068,0.000,0.002,0.012,0.000,-0.788580
3,1.0,0.0,0.0,0.0,0.0,1.478189,1.517227,1.537651,1.496265,0.457879,...,0.000,0.218,0.000,0.001,0.000,0.836,0.000,0.000,0.014,-1.138103
4,0.0,1.0,0.0,0.0,0.0,-0.344109,-0.177691,0.161098,-0.220213,-1.796328,...,0.000,0.017,0.000,0.000,0.000,0.000,0.000,0.000,0.122,0.434751
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14852,1.0,0.0,0.0,0.0,0.0,-2.241856,-0.872695,-0.278898,-0.705215,-1.451463,...,0.000,0.344,0.000,0.000,0.000,0.989,0.000,0.000,0.048,2.007605
14853,1.0,0.0,0.0,0.0,0.0,0.218550,0.117457,0.001146,0.141637,0.831918,...,0.047,0.000,0.510,0.002,0.000,0.000,0.999,0.322,0.000,1.658082
14854,0.0,0.0,0.0,1.0,0.0,-0.793832,-0.630800,-0.267386,-0.769272,-1.703582,...,0.118,0.000,0.993,0.000,0.001,0.000,0.542,0.146,0.000,0.347370
14855,1.0,0.0,0.0,0.0,0.0,0.100731,0.009953,-0.158611,-0.064708,0.207168,...,0.117,0.000,0.748,0.000,0.011,0.000,0.151,0.038,0.000,-0.351676


In [24]:
from sklearn.linear_model import LinearRegression

lin_reg = Pipeline([
    ('cleaning', preprocessing),
    ('linreg', LinearRegression())
])

lin_reg.fit(X_train, y_train)
y_pred = lin_reg.predict(X_test)

In [25]:
from sklearn.metrics import root_mean_squared_error

lin_rmse = root_mean_squared_error(y_test, y_pred)
lin_rmse

57494.299485082855

In [26]:
from sklearn.svm import SVR

svr = Pipeline([
    ("cleaning", preprocessing), 
    ("svr", SVR(kernel='rbf'))
    ])
svr.fit(X_train, y_train)
y_pred = svr.predict(X_test)
svr_rmse = root_mean_squared_error(y_test, y_pred)
svr_rmse

97705.2272888953

In [27]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("treereg", DecisionTreeRegressor())
    ])
tree_reg.fit(X_train, y_train)
y_pred = tree_reg.predict(X_test)
tree_rmse = root_mean_squared_error(y_test, y_pred)
tree_rmse

59718.64598676979

In [28]:
from sklearn.neighbors import KNeighborsRegressor

knn_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("knn_reg", KNeighborsRegressor())
    ])
knn_reg.fit(X_train, y_train)
y_pred = knn_reg.predict(X_test)
knn_rmse = root_mean_squared_error(y_test, y_pred)
knn_rmse

53460.46143634196

In [29]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", RandomForestRegressor())
    ])
forest_reg.fit(X_train, y_train)
y_pred = forest_reg.predict(X_test)
tree_rmse = root_mean_squared_error(y_test, y_pred)
tree_rmse

42889.604260087326

In [30]:
from sklearn.ensemble import VotingRegressor

voting_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", VotingRegressor([
		('lr', KNeighborsRegressor()),
		('rf', RandomForestRegressor(random_state=123))
    	])
	)
])
voting_reg.fit(X_train, y_train)
y_pred = voting_reg.predict(X_test)
voting_rmse = root_mean_squared_error(y_test, y_pred)
voting_rmse

45182.72504418528

In [31]:
from sklearn.ensemble import GradientBoostingRegressor

gb_reg = forest_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("forestreg", GradientBoostingRegressor(
        n_estimators=10,
        max_depth=10,)
    )
])
gb_reg.fit(X_train, y_train)
y_pred = gb_reg.predict(X_test)
gb_rmse = root_mean_squared_error(y_test, y_pred)
gb_rmse

54417.964413269474

In [32]:
from sklearn.ensemble import BaggingRegressor

bag_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("knn_reg", BaggingRegressor(
		KNeighborsRegressor(3), 
		n_estimators=500,
		max_samples=len(X_train) // 4, 
		bootstrap=True,
		n_jobs=-1, 
		random_state=123)
    )]
)
bag_reg.fit(X_train, y_train)
y_pred = bag_reg.predict(X_test)
bag_rmse = root_mean_squared_error(y_test, y_pred)
bag_rmse

52450.687814596444

In [33]:
from sklearn.ensemble import AdaBoostRegressor

adab_reg = Pipeline([
    ("cleaning", preprocessing), 
    ("adab_reg", AdaBoostRegressor(
		KNeighborsRegressor(3), 
		n_estimators=100,
		random_state=123)
    )]
)
adab_reg.fit(X_train, y_train)
y_pred = adab_reg.predict(X_test)
adab_rmse = root_mean_squared_error(y_test, y_pred)
adab_rmse

63084.420616335214

In [34]:
from sklearn.model_selection import cross_val_score

forest_rmses = -cross_val_score(
    forest_reg, 
    X.drop('income_cat', axis='columns', errors='ignore'), 
    y,
	scoring="neg_root_mean_squared_error", 
    cv=10)
forest_rmses, forest_rmses.mean()

(array([55650.14180803, 54443.2831176 , 55056.16068832, 55066.2438098 ,
        52572.65802087, 53122.82724633, 52061.51296824, 53046.48281165,
        54494.86872022, 54894.10341518]),
 np.float64(54040.828260623304))

In [35]:
voting_rmses = -cross_val_score(
    voting_reg, 
    X.drop('income_cat', axis='columns', errors='ignore'), 
    y,
	scoring="neg_root_mean_squared_error", 
    cv=10)
voting_rmses, voting_rmses.mean()

(array([46452.4227948 , 45122.79949797, 46008.09192118, 45355.45892701,
        43069.62976852, 43403.40479817, 43200.52278493, 43415.81392985,
        46260.37323861, 44559.2079774 ]),
 np.float64(44684.772563843515))